# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/amey05081999/Flyrank-Internship-Amey-Naik/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This notebook audits the Week-5 refresh-opportunity model with the same spirit used to read the FlyRank research paper: clarify where labels come from, test whether the validation design supports the claim, look for leakage, inspect real errors, and keep the wording proportional to the evidence.

**Public-safe scope:** this notebook uses the bundled anonymized dataset and does not display client names, URLs, domains, or private search queries.

## 1. Two paper findings + my methodology questions

### Finding 1 — “The Anatomy of Growing Content” (paper p. 6)

The paper reports that pages with rising impressions averaged **3.2K words and 184 days of age**, while pages with falling impressions averaged **2.3K words and 230 days**. It explicitly describes this as an observational comparison.

**Methodology question:** I would clarify exactly how the growing/declining label is constructed and whether the compared windows are temporally aligned. In particular, is “down” determined from the same 30-day-vs-previous-30-day performance window used to define the cohorts, and are all explanatory variables measured before that outcome window? If so, the result is a useful observed association; it should not be read as evidence that adding words or changing age will itself cause growth.

This is a strengthening question, not a challenge to the reported numbers: documenting the label provenance and feature timing would make the observational interpretation easier to reproduce.

### Finding 2 — “The Freshness Multiplier” (paper p. 9)

The paper reports that **365+ day content refreshed within 30 days showed 3.2× higher health (10.7 → 34.5) and 57× more impressions (71 → 4,039)**. It also notes that the 361+ freshness ratio is unstable because it contains only one declining page.

**Methodology question:** does the validation design support the stronger “refresh timing” interpretation, or is this a selected before/after cohort? Pages chosen for refresh may already differ from untouched pages in demand, editorial priority, or prior performance. A stronger design would compare refreshed pages with a clearly defined comparable control group, ideally with pre-refresh outcomes and a time-based holdout. That would separate an observed association from a causal claim.

The paper itself points readers toward comparing refreshed and unrevised pages over follow-up windows; making that comparison explicit would strengthen the evidence.

In [4]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import os

# Define the repository URL and expected path in Colab
repo_github_url = "https://github.com/amey05081999/Flyrank-Internship-Amey-Naik"
repo_name = "Flyrank-Internship-Amey-Naik"
repo_path = Path(f"/content/{repo_name}")

# Check if the repository is already cloned, if not, clone it
if not repo_path.exists():
    print(f"Cloning repository {repo_github_url} to {repo_path}...")
    os.system(f"git clone {repo_github_url} {repo_path}")
    print("Repository cloned successfully.")
else:
    print(f"Repository already exists at {repo_path}.")

# Set repo_root to the cloned repository path
repo_root = repo_path

# The raw_data_url is kept as a reference, but files will be accessed locally after cloning.
raw_data_url = "https://github.com/amey05081999/Flyrank-Internship-Amey-Naik/blob/main/data/raw/content_refresh_anonymized.csv"

# Add the scripts directory to sys.path to import ml_utils
sys.path.insert(0, str(repo_root / "scripts"))

from ml_utils import (
    MODEL_NUMERIC_FEATURES,
    MODEL_CATEGORICAL_FEATURES,
    precision_at_k,
)

RAW_PATH = repo_root / "data/raw/content_refresh_anonymized.csv"
FEATURE_PATH = repo_root / "data/processed/refresh_feature_vector.csv"

print("Repository root:", repo_root)
print("Raw dataset:", RAW_PATH)
print("Feature vector:", FEATURE_PATH)

# Verify that the RAW_PATH actually exists now
if not RAW_PATH.exists():
    raise FileNotFoundError(f"Data file not found at expected path: {RAW_PATH}")

Cloning repository https://github.com/amey05081999/Flyrank-Internship-Amey-Naik to /content/Flyrank-Internship-Amey-Naik...
Repository cloned successfully.
Repository root: /content/Flyrank-Internship-Amey-Naik
Raw dataset: /content/Flyrank-Internship-Amey-Naik/data/raw/content_refresh_anonymized.csv
Feature vector: /content/Flyrank-Internship-Amey-Naik/data/processed/refresh_feature_vector.csv


In [5]:
# Rebuild the feature vector from the bundled raw data so this audit does not depend on a hidden artifact.
import subprocess

subprocess.run(
    [sys.executable, str(repo_root / "scripts/01_prepare_features.py")],
    check=True,
    capture_output=True,
    text=True,
)
frame = pd.read_csv(FEATURE_PATH)

print(f"Rows available for modeling: {len(frame):,}")
print(f"Distinct clients: {frame['client_id'].nunique():,}")
print(f"Declining label rate (base rate): {frame['is_declining_label'].mean():.3f}")
print("Label definition:", "is_declining_label = (trend_direction == 'down')")

Rows available for modeling: 30,000
Distinct clients: 32
Declining label rate (base rate): 0.542
Label definition: is_declining_label = (trend_direction == 'down')


## 2. My model under an honest split (before/after)

The Week-5 pipeline's model family is logistic regression, a shallow decision tree, and random forest. The audit compares:

- **Before — row-level stratified split:** useful as an optimistic reference because the same client can appear in both train and test.
- **After — client-holdout split:** the test clients are unseen during training, matching the repeating-entity risk in this dataset.
- **After + leakage-safe features:** the client holdout is retained, while features that overlap the outcome window are removed.

The key point is that a grouped split alone does **not** fix temporal leakage. The label is derived from the most recent 30-day trend, so features containing that same 30-day window must also be treated as suspect.

In [6]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

RANDOM_STATE = 42

def make_matrix(df, numeric_features, categorical_features):
    nums = [c for c in numeric_features if c in df.columns]
    cats = [c for c in categorical_features if c in df.columns]
    numeric = (
        df[nums].apply(pd.to_numeric, errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
    )
    categorical = df[cats].fillna("unknown").astype(str)
    encoded = pd.get_dummies(
        categorical, prefix=cats, dummy_na=False, dtype=float
    )
    return pd.concat(
        [numeric.reset_index(drop=True), encoded.reset_index(drop=True)],
        axis=1,
    )

def build_rf():
    return RandomForestClassifier(
        class_weight="balanced_subsample",
        max_depth=10,
        min_samples_leaf=25,
        n_estimators=200,
        n_jobs=-1,
        random_state=RANDOM_STATE,
    )

def evaluate(y_true, probability):
    pred = (probability >= 0.5).astype(int)
    return {
        "accuracy": accuracy_score(y_true, pred),
        "precision": precision_score(y_true, pred, zero_division=0),
        "recall": recall_score(y_true, pred, zero_division=0),
        "f1": f1_score(y_true, pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, probability),
        "average_precision": average_precision_score(y_true, probability),
        "precision_at_50": precision_at_k(y_true, probability, 50),
    }

y = frame["is_declining_label"].astype(int)

# Before: row-level stratified split.
row_train, row_test = train_test_split(
    np.arange(len(frame)),
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

# After: deterministic client holdout, using the same 20% client rule as the Week-5 pipeline.
clients = frame["client_id"].fillna("unknown").astype(str).drop_duplicates().to_numpy()
rng = np.random.default_rng(RANDOM_STATE)
test_clients = set(rng.permutation(clients)[:max(1, round(len(clients) * 0.20))])
client_test_mask = frame["client_id"].fillna("unknown").astype(str).isin(test_clients)
client_test = np.where(client_test_mask)[0]
client_train = np.where(~client_test_mask)[0]

# Current Week-5 feature set.
X_current = make_matrix(frame, MODEL_NUMERIC_FEATURES, MODEL_CATEGORICAL_FEATURES)

rf_row = build_rf()
rf_row.fit(X_current.iloc[row_train], y.iloc[row_train])
row_prob = rf_row.predict_proba(X_current.iloc[row_test])[:, 1]

rf_client = build_rf()
rf_client.fit(X_current.iloc[client_train], y.iloc[client_train])
client_prob = rf_client.predict_proba(X_current.iloc[client_test])[:, 1]

# Leakage-safe feature set: only fields that do not contain the last-30-day outcome window.
safe_numeric = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
]
safe_categorical = [
    "competition_level", "content_type", "main_intent",
    "provider_used", "model_used",
]
X_safe = make_matrix(frame, safe_numeric, safe_categorical)

rf_safe = build_rf()
rf_safe.fit(X_safe.iloc[client_train], y.iloc[client_train])
safe_prob = rf_safe.predict_proba(X_safe.iloc[client_test])[:, 1]

comparison = pd.DataFrame([
    {"evaluation": "Before: row-level split + current features", **evaluate(y.iloc[row_test], row_prob)},
    {"evaluation": "After: client holdout + current features", **evaluate(y.iloc[client_test], client_prob)},
    {"evaluation": "After: client holdout + leakage-safe features", **evaluate(y.iloc[client_test], safe_prob)},
])

print(comparison.round(3).to_string(index=False))


                                   evaluation  accuracy  precision  recall    f1  roc_auc  average_precision  precision_at_50
   Before: row-level split + current features     0.692      0.709   0.732 0.721    0.758              0.768             0.90
     After: client holdout + current features     0.672      0.561   0.744 0.640    0.750              0.618             0.74
After: client holdout + leakage-safe features     0.781      0.716   0.730 0.723    0.861              0.727             0.80


### Before/after interpretation

The row-level result is not the preferred estimate because the same clients can contribute to both training and test data. The client-holdout result measures performance on clients not seen during training.

The final row in the table also removes features that overlap the label window. This is the more conservative validation result for the question being asked: **can the model flag decline using information available before the decline window?**

A useful audit outcome is therefore not “the score improved.” It is that the validation design now better matches the intended decision point and reduces the chance that the model is reading the answer from the outcome window.

## 3. Leakage audit

The target is `is_declining_label = (trend_direction == "down")`, where `trend_direction` is calculated from the 30-day impression change.

Three leakage risks were checked:

1. **Label-derived fields:** `trend_direction` and `trend_pct` are excluded from the model features.
2. **Future/overlapping windows:** `impressions_last_30d`, `clicks_last_30d`, and `sessions_last_30d` directly describe the outcome window; 90-day totals and rates also overlap it. These are excluded from the leakage-safe model.
3. **Decision-derived fields:** existing workflow/product flags and scores are not used as model inputs. The baseline score remains a comparator rather than a feature.

The important finding is that a grouped client split prevents cross-client memorization but does **not** by itself prevent temporal leakage.

In [7]:
# Explicit feature audit: print the suspect fields and confirm what the final model actually uses.
label_derived = {"trend_direction", "trend_pct", "is_declining_label"}
overlap_window = {
    "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d",
    "users_90d", "engaged_sessions_90d", "ai_sessions_90d",
    "scroll_events_90d", "days_with_impressions", "days_with_sessions",
    "ctr", "engagement_rate", "scroll_rate", "ai_traffic_pct",
    "log_impressions_90d", "log_clicks_90d", "log_sessions_90d",
    "log_ai_sessions_90d", "has_clicks", "has_ai_sessions",
    "measurable_opportunity", "avg_position",
    "content_age_days", "days_since_last_update",
    "age_tier", "freshness_tier", "word_count_tier",
    "impression_tier", "position_tier",
}
decision_derived_keywords = {"score", "flag", "action", "reason"}

current_feature_names = set(MODEL_NUMERIC_FEATURES + MODEL_CATEGORICAL_FEATURES)
safe_feature_names = set(safe_numeric + safe_categorical)

audit_rows = []
for feature in sorted(current_feature_names):
    if feature in label_derived:
        risk = "label-derived"
        decision = "excluded"
    elif feature in overlap_window:
        risk = "outcome-window overlap"
        decision = "excluded from leakage-safe model"
    elif any(k in feature.lower() for k in decision_derived_keywords):
        risk = "possible decision-derived field"
        decision = "excluded"
    else:
        risk = "no direct leakage found from this taxonomy"
        decision = "eligible"

    audit_rows.append({"feature": feature, "risk": risk, "decision": decision})

audit_table = pd.DataFrame(audit_rows)

print("Label-derived features present in current feature list:",
      sorted(current_feature_names & label_derived))
print("Outcome-window features removed from safe model:",
      sorted((current_feature_names & overlap_window)))
print("Safe feature count:", len(safe_feature_names))
print("Safe features:", sorted(safe_feature_names))

audit_table

Label-derived features present in current feature list: []
Outcome-window features removed from safe model: ['age_tier', 'ai_traffic_pct', 'avg_position', 'content_age_days', 'ctr', 'days_since_last_update', 'days_with_impressions', 'days_with_sessions', 'engagement_rate', 'freshness_tier', 'impression_tier', 'log_ai_sessions_90d', 'log_clicks_90d', 'log_impressions_90d', 'log_sessions_90d', 'position_tier', 'scroll_rate', 'word_count_tier']
Safe feature count: 13
Safe features: ['char_count', 'clicks_prev_30d', 'competition', 'competition_level', 'content_type', 'cpc', 'impressions_prev_30d', 'main_intent', 'model_used', 'provider_used', 'search_volume', 'sessions_prev_30d', 'word_count']


,feature,risk,decision
0,age_tier,outcome-window overlap,excluded from leakage-safe model
1,ai_traffic_pct,outcome-window overlap,excluded from leakage-safe model
2,avg_position,outcome-window overlap,excluded from leakage-safe model
3,char_count,no direct leakage found from this taxonomy,eligible
4,competition,no direct leakage found from this taxonomy,eligible
5,competition_level,no direct leakage found from this taxonomy,eligible
6,content_age_days,outcome-window overlap,excluded from leakage-safe model
7,content_type,no direct leakage found from this taxonomy,eligible
8,cpc,no direct leakage found from this taxonomy,eligible
9,ctr,outcome-window overlap,excluded from leakage-safe model


### Why the overlap matters

For this dataset, the outcome is defined using the latest 30 days versus the preceding 30 days. A feature such as `impressions_last_30d` is therefore not merely correlated with the label — it is part of the calculation from which the label is created. The same concern extends to 90-day aggregates because they contain that latest 30-day period.

The leakage-safe model instead uses the previous 30-day window plus metadata available independently of the outcome window. This does not prove production readiness; it is a more defensible measurement of the stated prediction task.

## 4. Claim rewrite

### Original-style claim

> “The random forest accurately predicts declining content and can be used to prioritize refresh decisions.”

### Evidence-matched rewrite

> “On this anonymized dataset, the random forest **measured** out-of-sample ranking performance under a client-holdout split. After removing features that overlap the decline-label window, the model **observed** a precision@50 of approximately the value reported below. These results provide **directional decision-support** for review prioritization; they do not establish causal effects of refreshing content or guarantee performance on future clients or production data.”

The safer wording matters because the evaluation is observational and based on one bundled dataset. It supports a measured ranking/flagging statement, not a causal claim about what a refresh will do.

In [ ]:
# Report the headline number used by the rewritten claim.
safe_metrics = evaluate(y.iloc[client_test], safe_prob)
print(f"Leakage-safe client-holdout precision@50: {safe_metrics['precision_at_50']:.3f}")
print(f"Leakage-safe client-holdout ROC AUC: {safe_metrics['roc_auc']:.3f}")
print(f"Overall label base rate: {y.mean():.3f}")

## 4b. Real failure examples

The examples below come from the leakage-safe client-holdout test set. They deliberately omit identifiers and show only model-relevant, non-private fields.

A failure is useful here because it tests whether the model's mistakes are plausible rather than hiding them behind an aggregate metric.

In [8]:
# Inspect five high-confidence mistakes from the leakage-safe test set.
test_frame = frame.iloc[client_test].copy()
test_frame["predicted_probability"] = safe_prob
test_frame["predicted_label"] = (safe_prob >= 0.5).astype(int)
test_frame["actual_label"] = y.iloc[client_test].to_numpy()
test_frame["confidence_gap"] = (test_frame["predicted_probability"] - 0.5).abs()

failures = test_frame[
    test_frame["predicted_label"] != test_frame["actual_label"]
].sort_values("confidence_gap", ascending=False).head(5).copy()

failure_view = failures[
    [
        "actual_label", "predicted_label", "predicted_probability",
        "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
        "word_count", "content_type"
    ]
].reset_index(drop=True)

failure_view.round(3)

,actual_label,predicted_label,predicted_probability,impressions_prev_30d,clicks_prev_30d,sessions_prev_30d,word_count,content_type
0,0,1,0.817,442,1,1,4095.0,keyword article
1,0,1,0.790,919,0,5,3546.0,keyword article
2,0,1,0.790,738,1,2,3444.0,keyword article
3,0,1,0.784,220,1,2,3281.0,keyword article
4,0,1,0.783,9039,2,5,3251.0,keyword article


### Failure interpretation

These are classification/ranking errors, not evidence that the underlying content is “bad.” The model sees a limited set of historical performance and metadata signals, while the label compresses a future change into a binary outcome. Pages with similar historical signals can therefore move in different directions.

The practical implication is to use the score as a **review queue / decision-support signal**, with human inspection and current editorial context before action.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.